In [19]:
import os
from datetime import datetime

from reportlab.lib.pagesizes import A4
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image as RLImage
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.platypus.flowables import HRFlowable

# Configuration dictionary (CFG) and report directory
CFG = {
    'report_dir': '/content/reports'
}

# Create the report directory if it doesn't exist
os.makedirs(CFG['report_dir'], exist_ok=True)

In [20]:
def make_pdf(img_path, cls, is_mal, conf, all_probs, answers, assessment, ents):
    ts  = datetime.now().strftime('%Y%m%d_%H%M%S')
    out = os.path.join(CFG['report_dir'], f'report_{ts}.pdf')
    doc = SimpleDocTemplate(out,pagesize=A4,
                             topMargin=0.75*inch,bottomMargin=0.75*inch,
                             leftMargin=0.9*inch,rightMargin=0.9*inch)
    sty = getSampleStyleSheet()
    H2  = ParagraphStyle('H2', parent=sty['Heading2'],fontSize=13,spaceAfter=4,spaceBefore=10)
    BDY = ParagraphStyle('BDY',parent=sty['Normal'],  fontSize=10,spaceAfter=4,leading=14)
    CAP = ParagraphStyle('CAP',parent=sty['Normal'],  fontSize=8, textColor=colors.grey)
    WRN = ParagraphStyle('WRN',parent=sty['Normal'],  fontSize=10,textColor=colors.red,spaceAfter=4)

    story=[
        Paragraph('Skin Cancer AI Screening Report',sty['Title']),
        Paragraph(datetime.now().strftime('%B %d, %Y %H:%M'),CAP),
        HRFlowable(width='100%',thickness=1,color=colors.grey),
        Spacer(1,0.1*inch),
        Paragraph('DISCLAIMER: Research/screening only. NOT a medical diagnosis. Consult a dermatologist.',WRN),
        Spacer(1,0.1*inch),
    ]
    if img_path and os.path.exists(img_path):
        story+=[Paragraph('Input Image',H2),RLImage(img_path,2.5*inch,2.5*inch),Spacer(1,0.1*inch)]

    story.append(Paragraph('AI Classification Result',H2))
    td=[['Parameter','Value'],
        ['Detected Condition',cls],
        ['Malignancy','POTENTIALLY MALIGNANT' if is_mal else 'LIKELY BENIGN'],
        ['Confidence',f'{conf:.1%}'],
        ['Model','ConvNeXtV2 + ViT-B/16 + Mamba Fusion']]
    t=Table(td,colWidths=[2.5*inch,3.5*inch])
    t.setStyle(TableStyle([
        ('BACKGROUND',(0,0),(-1,0),colors.HexColor('#2C3E50')),
        ('TEXTCOLOR',(0,0),(-1,0),colors.white),
        ('FONTNAME',(0,0),(-1,0),'Helvetica-Bold'),
        ('FONTSIZE',(0,0),(-1,-1),10),
        ('ROWBACKGROUNDS',(0,1),(-1,-1),[colors.white,colors.HexColor('#F2F2F2')]),
        ('GRID',(0,0),(-1,-1),0.5,colors.grey),
        ('LEFTPADDING',(0,0),(-1,-1),8),('TOPPADDING',(0,0),(-1,-1),5),
    ]))
    story+=[t,Spacer(1,0.1*inch)]

    story.append(Paragraph('Class Probabilities',H2))
    pd2=[['Condition','Probability']]+[[k,f'{v:.2%}'] for k,v in sorted(all_probs.items(),key=lambda x:-x[1])]
    pt=Table(pd2,colWidths=[3.5*inch,2.0*inch])
    pt.setStyle(TableStyle([
        ('BACKGROUND',(0,0),(-1,0),colors.HexColor('#2980B9')),
        ('TEXTCOLOR',(0,0),(-1,0),colors.white),
        ('FONTNAME',(0,0),(-1,0),'Helvetica-Bold'),
        ('FONTSIZE',(0,0),(-1,-1),9),
        ('ROWBACKGROUNDS',(0,1),(-1,-1),[colors.white,colors.HexColor('#EBF5FB')]),
        ('GRID',(0,0),(-1,-1),0.5,colors.lightgrey),
        ('LEFTPADDING',(0,0),(-1,-1),8),('TOPPADDING',(0,0),(-1,-1),4),
    ]))
    story+=[pt,Spacer(1,0.1*inch)]

    story.append(Paragraph('Clinical Interview',H2))
    for q,a in answers.items():
        story.append(Paragraph(f'<b>Q:</b> {q}',BDY))
        story.append(Paragraph(f'<b>A:</b> {a}',BDY))

    if ents:
        story.append(Paragraph('Medical Entities (NER)',H2))
        story.append(Paragraph(', '.join(sorted(set(e['word'] for e in ents[:15]))),BDY))

    story+=[Paragraph('AI Clinical Assessment (RAG-Grounded)',H2),
            Paragraph('Generated from clinical knowledge base. Not a diagnosis.',CAP)]
    for line in assessment.split('\n'):
        if line.strip(): story.append(Paragraph(line.strip(),BDY))

    story+=[HRFlowable(width='100%',thickness=0.5,color=colors.grey),
            Paragraph('AI skin cancer screening. Always consult a qualified dermatologist.',CAP)]
    doc.build(story)
    print(f'PDF saved: {out}')
    return out

In [21]:
import torch
import numpy as np
import faiss
import re
from pypdf import PdfReader
from transformers import AutoTokenizer, AutoModel, pipeline, logging as hf_logging

In [22]:
from dotenv import load_dotenv
load_dotenv()
hf_logging.set_verbosity_error()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
BIOBERT_MODEL = "dmis-lab/biobert-base-cased-v1.1"

from transformers import BertTokenizer, BertModel
tokenizer = BertTokenizer.from_pretrained(BIOBERT_MODEL)
biobert = BertModel.from_pretrained(BIOBERT_MODEL).to(DEVICE)
biobert.eval()

Using device: cuda


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(28996, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [23]:
NER_OK=False
try:
    ner = pipeline('ner',model='allenai/scibert_scivocab_uncased',
                      tokenizer='allenai/scibert_scivocab_uncased',
                      aggregation_strategy='simple',
                      device=0 if torch.cuda.is_available() else -1)
    NER_OK=True; print('NER loaded')
except Exception as e: print(f'NER fallback ({e})')

MED=['melanoma','carcinoma','lesion','tumor','biopsy','metastasis',
     'dermoscopy','excision','immunotherapy','ulcer','nodule','mole',
     'keratosis','dysplastic','atypical','benign','malignant','erythema']

def get_ents(text):
    if NER_OK:
        try: return [{'word':e['word'],'label':e['entity_group']} for e in ner(text)]
        except: pass
    return [{'word':t,'label':'MED'} for t in MED if t in text.lower()]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

NER loaded


In [24]:
def encode_text(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(DEVICE)

    with torch.no_grad():
        outputs = biobert(**inputs)

    return outputs.last_hidden_state[:, 0, :].cpu().numpy()

In [25]:
def semantic_chunk_pdf(pdf_path):
    reader = PdfReader(pdf_path)
    text = "\n".join([page.extract_text() for page in reader.pages])

    chunks = re.split(
        r"\n(?=[A-Z][A-Za-z\s]{3,}:|\d+\.\s)",
        text
    )

    return [c.strip() for c in chunks if len(c.strip()) > 200]

In [26]:
def build_vector_store(chunks):
    embeddings = np.vstack([encode_text(c) for c in chunks])
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)
    return index, chunks

In [27]:
def load_disease_kb(pdf_path):
    chunks = semantic_chunk_pdf(pdf_path)
    index, texts = build_vector_store(chunks)
    return index, texts

In [28]:
def retrieve_context(query, index, texts, top_k=3):
    query_emb = encode_text(query)
    _, indices = index.search(query_emb, top_k)
    return "\n".join([texts[i] for i in indices[0]])

In [37]:
QUESTION_BANK = {
    "MEL": [  # Melanoma
        "Is the lesion asymmetrical?",
        "Does it have irregular or poorly defined borders?",
        "Is there variation in color within the lesion?",
        "Has the lesion changed in size over time?",
        "Does it itch, bleed, or fail to heal?",
        "Is the lesion larger than 6mm in diameter?",
        "Is there a history of sun exposure or sunburns?",
        "Does the lesion show any shades of black, blue, or red?",
        "Has the lesion evolved or changed recently?",
        "Is there a family history of melanoma?"
    ],
    "DF": [  # Dermatofibroma
        "Is the lesion firm and dome-shaped?",
        "Does it dimple inward when pinched?",
        "Has it remained stable in size?",
        "Is it painless or only mildly tender?",
        "Did it appear after minor skin trauma?",
        "Is the lesion less than 1cm in diameter?",
        "Is the color brown, reddish-brown, or pink?",
        "Is the lesion fixed to the skin surface but movable over deeper tissues?",
        "Is the lesion on the legs or arms?",
        "Does the lesion feel rubbery to the touch?"
    ],
    "AK": [  # Actinic Keratosis
        "Is the lesion rough and scaly?",
        "Does it feel like sandpaper or crusty?",
        "Is it located on sun-exposed skin?",
        "Has it persisted despite treatment?",
        "Is there surrounding skin damage or sun damage?",
        "Is the lesion flat or slightly raised?",
        "Is the color flesh-colored, pink, red, or brown?",
        "Does the lesion itch or burn?",
        "Is there a history of sun exposure?",
        "Does the lesion bleed easily?"
    ],
    "BCC": [  # Basal Cell Carcinoma
        "Does the lesion have a pearly or waxy appearance?",
        "Does it have visible blood vessels (telangiectasia)?",
        "Is there an ulcer or open sore that doesn't heal?",
        "Is it located on sun-exposed skin?",
        "Does it bleed easily?",
        "Is the lesion pink, red, or flesh-colored?",
        "Does the lesion have a raised, rolled border?",
        "Is the lesion slow-growing?",
        "Is there any crusting or oozing?",
        "Is there a history of sun exposure?"
    ],
    "BKL": [  # Benign Keratosis-like lesions
        "Does the lesion have a waxy or 'stuck-on' appearance?",
        "Is the lesion well-demarcated?",
        "Are there multiple similar lesions elsewhere?",
        "Does it have a rough or warty texture?",
        "Is it brown, black, or yellowish in color?",
        "Does the lesion have comedones or milia-like cysts?",
        "Has the lesion been stable for years?",
        "Is the lesion round or oval?",
        "Is the lesion usually asymptomatic?",
        "Does the lesion have a velvety surface?"
    ],
    "NV": [  # Melanocytic Nevus (Mole)
        "Is the lesion round and symmetrical?",
        "Does it have regular, well-defined borders?",
        "Is it uniformly colored (brown or tan)?",
        "Has it been present since childhood or young adulthood?",
        "Is it smaller than 6mm in diameter?",
        "Is the lesion usually asymptomatic?",
        "Is the lesion flat or slightly raised?",
        "Does the lesion have a smooth surface?",
        "Are there multiple similar lesions elsewhere?",
        "Has the lesion been stable for years?"
    ],
    "SCC": [  # Squamous Cell Carcinoma
        "Is the lesion rough, scaly, or warty?",
        "Does it have a crust or ulcer?",
        "Is it tender or painful?",
        "Is it located on sun-exposed skin?",
        "Does it bleed easily?",
        "Is there surrounding induration or erythema?",
        "Is the lesion growing rapidly?",
        "Is there a history of actinic keratosis?",
        "Is the lesion firm to the touch?",
        "Does the lesion have a raised border?"
    ],
    "VASC": [  # Vascular lesions
        "Is the lesion red, purple, or blue?",
        "Does it blanch with pressure?",
        "Is it soft and compressible?",
        "Is it usually asymptomatic?",
        "Is the lesion flat or raised?",
        "Is the lesion usually painless?",
        "Does the lesion have a vascular appearance?",
        "Is there a history of trauma?",
        "Is the lesion located on the face, trunk, or neck?",
        "Has the lesion been stable for years?"
    ]
}

In [31]:
def score_answer(answer, context):
    combined = answer + " " + context
    embedding = encode_text(combined)
    return float(np.linalg.norm(embedding))

In [32]:
def symptom_assessment(disease, index, texts):
    scores = []
    user_answers = {}

    print(f"\nSymptom Assessment for {disease.upper()}\n")

    for question in QUESTION_BANK[disease]:
        answer = input(question + "\n> ")
        user_answers[question] = answer # Store the answer
        context = retrieve_context(question, index, texts)
        score = score_answer(answer, context)
        scores.append(score)

    return np.mean(scores), user_answers # Return both the score and answers

In [33]:
def normalize_score(score):
    return 1 / (1 + np.exp(-score))

In [34]:
def fuse_predictions(image_prob, nlp_score, alpha=0.7):
    return alpha * image_prob + (1 - alpha) * nlp_score

In [43]:
# Example output from your image classifier
predicted_class = "VASC"
image_probability = 0.76

rag_file_paths = {
    "AK": "rag/ACTINIC KERATOSIS.pdf",
    "BCC": "rag/BASAL CELL CARCINOMA.pdf",
    "BKL": "rag/BENIGN KERATOSIS.pdf",
    "DF": "rag/DERMATOFIBROMA.pdf",
    "MEL": "rag/MELANOMA.pdf",
    "NV": "rag/MELANOCYTIC NEVUS.pdf",
    "SCC": "rag/SQUAMOUS CELL CARCINOMA.pdf",
    "VASC": "rag/VASCULAR LESIONS.pdf"
}

# Load disease-specific PDF knowledge base
# pdf_path = "MELANOMA.pdf"  # change dynamically per class
pdf_path = rag_file_paths[predicted_class]
print( pdf_path )
index, texts = load_disease_kb(pdf_path)

# Run NLP pipeline
symptom_score_raw, user_answers_ignored = symptom_assessment(predicted_class, index, texts)
symptom_probability = normalize_score(symptom_score_raw)

# Final prediction
final_probability = fuse_predictions(
    image_probability,
    symptom_probability
)

print("\nFINAL RESULTS")
print(f"Image Model Confidence: {image_probability:.3f}")
print(f"Symptom-Based Confidence: {symptom_probability:.3f}")
print(f"Final Fused Confidence: {final_probability:.3f}")

print("\nFor research use only. Not a clinical diagnosis.")

rag/VASCULAR LESIONS.pdf

Symptom Assessment for VASC



Is the lesion red, purple, or blue?
>  no
Does it blanch with pressure?
>  yes
Is it soft and compressible?
>  no
Is it usually asymptomatic?
>  yes
Is the lesion flat or raised?
>  yes
Is the lesion usually painless?
>  yes
Does the lesion have a vascular appearance?
>  yes
Is there a history of trauma?
>  yes
Is the lesion located on the face, trunk, or neck?
>  no
Has the lesion been stable for years?
>  no



FINAL RESULTS
Image Model Confidence: 0.760
Symptom-Based Confidence: 1.000
Final Fused Confidence: 0.832

For research use only. Not a clinical diagnosis.


In [36]:
# Prepare data for the PDF report
img_path = None # Assuming no image is currently processed via a file path
cls = predicted_class
is_mal = (predicted_class == "MEL") # Simplified: Melanoma is generally considered malignant
conf = final_probability

# Construct all_probs based on image_probability for simplicity
all_probs = {predicted_class: image_probability}
other_classes = [c for c in QUESTION_BANK.keys() if c != predicted_class]
if other_classes:
    remaining_prob = (1 - image_probability) / len(other_classes)
    for oc in other_classes: all_probs[oc] = remaining_prob

# Placeholder for assessment and entities, as they are not yet generated in the notebook
# assessment = "This is a preliminary AI clinical assessment based on the provided symptoms and knowledge base. Further medical consultation is highly recommended. The AI has identified certain patterns consistent with the predicted class, but this is not a diagnostic tool."

# Use NER to get entities from user answers
all_answers_text = " ".join(user_answers_ignored.values())
ents = get_ents(all_answers_text)

# Generate assessment based on entities and context (this is a placeholder for actual RAG-Grounded assessment)
assessment = "This is a preliminary AI clinical assessment based on the provided symptoms and knowledge base. Further medical consultation is highly recommended. The AI has identified certain patterns consistent with the predicted class, but this is not a diagnostic tool." # Replace with actual RAG-generated assessment if available

# Re-run symptom_assessment to capture answers (already done above, just referencing user_answers_ignored)
# symptom_score_raw, user_answers = symptom_assessment(predicted_class, index, texts)
# symptom_probability = normalize_score(symptom_score_raw)

# Call the make_pdf function
report_file = make_pdf(img_path, cls, is_mal, conf, all_probs, user_answers_ignored, assessment, ents)
print(f"Report generated at: {report_file}")

PDF saved: /content/reports\report_20260630_222325.pdf
Report generated at: /content/reports\report_20260630_222325.pdf
